# PegaSUS DATASUS pipeline notebook

This notebook rebuilds the DATASUS pipeline **from the frozen FTP scan artifact onward**.

It is structured around a small number of operational steps:

1. Configure paths and output locations.
2. Define deterministic heuristics and parsing logic.
3. Load `datasus_scan.jsonl` and build a normalized asset inventory.
4. Build logical data families and associate document assets.
5. Audit, summarize, and export artifacts.
6. Drill specific families and failure modes interactively.

Key redesign points in this notebook:

- it does **not** collapse state-partitioned families into one representative file per period;
- it separates **partition type** from **coverage completeness**;
- it supports multiple filename grammars, including:
  - `[prefix][UF|BR][YYMM]`
  - `[prefix][UF|BR][YY]`
  - `[prefix][UF|BR][YYYY]`
  - `[prefix][YYMM]`
  - `[prefix][YY]`
  - `[prefix][YYYY]`
- it treats documentation as a broader **document asset catalog**, not just PDFs.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "pegasus_data").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate project root. Expected a parent directory containing "
        "'src/pegasus_data' and 'data'."
    )

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
CATALOG_ROOT = DATA_ROOT / "catalog"
OUTPUT_ROOT = CATALOG_ROOT / "notebook_experiments"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SCAN_JSONL = CATALOG_ROOT / "datasus_scan.jsonl"

INVENTORY_OUT = OUTPUT_ROOT / "datasus_inventory_notebook.jsonl"
DOC_ASSETS_OUT = OUTPUT_ROOT / "datasus_doc_assets_notebook.json"
FAMILIES_OUT = OUTPUT_ROOT / "datasus_families_notebook.json"
SUMMARY_OUT = OUTPUT_ROOT / "datasus_families_summary_notebook.json"
AUDIT_OUT = OUTPUT_ROOT / "datasus_family_audit_notebook.json"

NOTEBOOK_ROOT = OUTPUT_ROOT

INVENTORY_JSONL = INVENTORY_OUT
DOC_ASSETS_JSON = DOC_ASSETS_OUT
FAMILIES_JSON = FAMILIES_OUT
FAMILY_SUMMARY_JSON = SUMMARY_OUT
FAMILY_AUDIT_JSON = AUDIT_OUT
COMPACT_SUMMARY_JSON = OUTPUT_ROOT / "datasus_run_compact_summary.json"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SCAN_JSONL   =", SCAN_JSONL)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)

PROJECT_ROOT = C:\Users\Galaxy\LEVI\projects\PegaSUS
SCAN_JSONL   = C:\Users\Galaxy\LEVI\projects\PegaSUS\data\catalog\datasus_scan.jsonl
OUTPUT_ROOT  = C:\Users\Galaxy\LEVI\projects\PegaSUS\data\catalog\notebook_experiments


In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any


PUBLIC_ROOT = "/dissemin/publicos"
UF_CODES = {
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS",
    "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC",
    "SE", "SP", "TO",
}
NATIONAL_CODES = {"BR"}
KNOWN_GEO_CODES = UF_CODES | NATIONAL_CODES

STRUCTURED_EXTENSIONS = {".dbc", ".dbf", ".json", ".xml", ".csv", ".parquet"}
WRAPPER_EXTENSIONS = {".zip", ".gz"}
DOC_EXTENSIONS = {".pdf", ".txt", ".md", ".doc", ".docx", ".xls", ".xlsx", ".csv", ".zip"}
FORMAT_DIR_HINTS = {"JSON", "XML", "CSV", "PARQUET"}
DOC_DIR_TOKENS = {"DOC", "DOCS", "DOCUMENTO", "DOCUMENTOS", "MANUAL", "LEIA", "README", "LAYOUT"}
AUX_DIR_TOKENS = {"AUX", "AUXILIAR", "AUXILIARES", "TAB", "TABELA", "TABELAS", "TABWIN"}
DATA_DIR_TOKENS = {"DADOS", "DADO", "DADOS_", "DADOS-"}
NOISE_DIR_PATTERNS = (
    re.compile(r"^\d{6}_?$"),
    re.compile(r"^\d{4}_\d{4}$"),
    re.compile(r"^CID\d+$", re.I),
)
KEYWORD_DOC_HINTS = {"dicionario", "dic_dados", "layout", "manual", "instrucao", "instrucoes", "estrutura", "readme", "nota", "docs_tab"}
YEAR_MIN = 1970
YEAR_MAX = 2035
FORMAT_PREFERENCE = {".json": 0, ".parquet": 1, ".xml": 2, ".csv": 3, ".dbf": 4, ".dbc": 5}

PATH_SEMANTIC_TOKENS = {"FINAIS", "FINAL", "PRELIM", "PRELIMINAR", "HOMOLOG", "HOMOLOGACAO", "TESTE", "TESTES"}
DATE_KIND_RANK = {"yymm": 0, "yyyy": 1, "yy": 2}
PATH_SEMANTIC_RANK = {"[Primary]": 0, "[Legacy]": 1, "[Staging]": 2, "[Test]": 3}
DOC_GENERIC_HINTS = {"TAB", "TABS", "TABWIN", "DICIONARIO", "DIC", "LAYOUT", "MANUAL", "ESTRUTURA", "INFORME", "NOTA", "DOCS"}
TOKEN_SPLIT_RE = re.compile(r"[^A-Z0-9]+")

def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_json(path: str | Path, payload: Any) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_jsonl(path: str | Path, rows: list[dict[str, Any]]) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def suffix_chain(path: str) -> list[str]:
    return [suffix.lower() for suffix in PurePosixPath(path).suffixes]


def strip_all_suffixes(name: str) -> str:
    base = PurePosixPath(name).name
    for suffix in reversed(PurePosixPath(name).suffixes):
        if base.lower().endswith(suffix.lower()):
            base = base[:-len(suffix)]
    return base


def primary_extension(path: str) -> str | None:
    suffixes = suffix_chain(path)
    for suffix in reversed(suffixes):
        if suffix in STRUCTURED_EXTENSIONS:
            return suffix
    upper = path.upper()
    for token, ext in (("/JSON/", ".json"), ("/XML/", ".xml"), ("/CSV/", ".csv"), ("/PARQUET/", ".parquet")):
        if token in upper:
            return ext
    return suffixes[-1] if suffixes else None


def format_family(path: str) -> str:
    ext = primary_extension(path)
    return {
        ".dbc": "dbase",
        ".dbf": "dbase",
        ".json": "json",
        ".xml": "xml",
        ".csv": "csv",
        ".parquet": "parquet",
        ".pdf": "pdf",
        ".zip": "archive",
        ".gz": "archive",
    }.get(ext or "", "unknown")


def normalize_public_path(path: str) -> str:
    if path.startswith("ftp://"):
        path = PurePosixPath(re.sub(r"^ftp://[^/]+", "", path)).as_posix()
    return path if path.startswith("/") else "/" + path


def split_public_parts(path: str) -> list[str]:
    return [part for part in PurePosixPath(normalize_public_path(path)).parts if part != "/"]


def infer_system_context(path: str) -> dict[str, Any]:
    normalized = normalize_public_path(path)
    directory = str(PurePosixPath(normalized).parent)
    parts = split_public_parts(directory)

    if "publicos" not in parts:
        return {
            "distribution_root": None,
            "system": None,
            "subsystem": None,
            "context_parts": [],
            "meaningful_context_parts": [],
        }

    idx = parts.index("publicos")
    tail = parts[idx + 1:]
    if not tail:
        return {
            "distribution_root": None,
            "system": None,
            "subsystem": None,
            "context_parts": [],
            "meaningful_context_parts": [],
        }

    distribution_root = tail[0]
    if distribution_root == "Dados_Abertos" and len(tail) >= 2:
        system = tail[1]
        context_tail = tail[2:]
    else:
        system = distribution_root
        context_tail = tail[1:]

    meaningful = []
    for token in context_tail:
        upper = token.upper()
        if upper in FORMAT_DIR_HINTS | DOC_DIR_TOKENS | AUX_DIR_TOKENS | DATA_DIR_TOKENS | PATH_SEMANTIC_TOKENS:
            continue
        if any(pattern.match(token) for pattern in NOISE_DIR_PATTERNS):
            continue
        meaningful.append(token)

    subsystem = meaningful[-1] if meaningful else None
    return {
        "distribution_root": distribution_root,
        "system": system,
        "subsystem": subsystem,
        "context_parts": context_tail,
        "meaningful_context_parts": meaningful,
    }


def classify_asset(path: str) -> str:
    ext = primary_extension(path) or ""
    upper_parts = [part.upper() for part in split_public_parts(path)]
    filename = PurePosixPath(path).name.lower()
    has_doc_dir = any(token in DOC_DIR_TOKENS for token in upper_parts)
    has_aux_dir = any(token in AUX_DIR_TOKENS for token in upper_parts)
    has_data_dir = any(token in DATA_DIR_TOKENS or token in FORMAT_DIR_HINTS for token in upper_parts)
    has_doc_keyword = any(keyword in filename for keyword in KEYWORD_DOC_HINTS)

    if ext in STRUCTURED_EXTENSIONS:
        return "data"
    if ext in {".zip", ".gz"}:
        if has_doc_dir or has_doc_keyword:
            return "doc"
        if has_data_dir:
            return "data"
    if ext in DOC_EXTENSIONS and (has_doc_dir or has_aux_dir or has_doc_keyword):
        return "doc"
    if has_aux_dir:
        return "aux"
    if has_doc_dir:
        return "doc"
    return "other"


def normalize_period(date_token: str, date_kind: str) -> int | None:
    try:
        if date_kind == "yymm":
            yy = int(date_token[:2])
            mm = int(date_token[2:])
            if not 1 <= mm <= 12:
                return None
            year = 2000 + yy if yy < 70 else 1900 + yy
            return year * 100 + mm
        if date_kind == "yy":
            yy = int(date_token)
            year = 2000 + yy if yy < 70 else 1900 + yy
            return year * 100
        if date_kind == "yyyy":
            year = int(date_token)
            if not YEAR_MIN <= year <= YEAR_MAX:
                return None
            return year * 100
    except Exception:
        return None
    return None


def period_label(value: int | None) -> str | None:
    if value is None:
        return None
    year, month = divmod(value, 100)
    if month == 0:
        return str(year)
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    if 1 <= month <= 12:
        return f"{month_names[month - 1]} {year}"
    return f"{year}-{month:02d}"


def next_period(value: int, date_kind: str) -> int:
    year, month = divmod(value, 100)
    if date_kind == "yymm":
        month += 1
        if month > 12:
            year += 1
            month = 1
        return year * 100 + month
    return (year + 1) * 100


def expected_periods(start: int | None, end: int | None, date_kind: str) -> list[int]:
    if start is None or end is None:
        return []
    out = []
    current = start
    guard = 0
    while current <= end and guard < 5000:
        out.append(current)
        current = next_period(current, date_kind)
        guard += 1
    return out


def candidate_parses(stem: str) -> list[dict[str, Any]]:
    candidates = []
    upper = stem.upper()

    def emit(prefix: str, geo_code: str | None, date_token: str, date_kind: str, uses_geo: bool, source: str) -> None:
        period = normalize_period(date_token, date_kind)
        if period is None:
            return
        if len(prefix) < 2:
            return
        score = {"yymm": 50, "yyyy": 35, "yy": 25}[date_kind]
        score += 30 if uses_geo else 10
        score += 15 if 2 <= len(prefix) <= 8 else 0
        score += 10 if prefix.isalpha() else 0
        candidates.append({
            "prefix": prefix,
            "geo_code": geo_code,
            "date_token": date_token,
            "date_kind": date_kind,
            "period": period,
            "uses_geo": uses_geo,
            "score": score,
            "source": source,
        })

    if len(upper) >= 6 and upper[-4:].isdigit():
        date_token = upper[-4:]
        rem = upper[:-4]
        mm = int(date_token[2:])
        if 1 <= mm <= 12:
            if len(rem) >= 4 and rem[-2:] in KNOWN_GEO_CODES:
                emit(rem[:-2], rem[-2:], date_token, "yymm", True, "suffix4_geo")
            emit(rem, None, date_token, "yymm", False, "suffix4_nogeo")
        yyyy = int(date_token)
        if YEAR_MIN <= yyyy <= YEAR_MAX:
            if len(rem) >= 4 and rem[-2:] in KNOWN_GEO_CODES:
                emit(rem[:-2], rem[-2:], date_token, "yyyy", True, "suffix4_geo")
            emit(rem, None, date_token, "yyyy", False, "suffix4_nogeo")

    if len(upper) >= 4 and upper[-2:].isdigit():
        date_token = upper[-2:]
        rem = upper[:-2]
        if len(rem) >= 4 and rem[-2:] in KNOWN_GEO_CODES:
            emit(rem[:-2], rem[-2:], date_token, "yy", True, "suffix2_geo")
        emit(rem, None, date_token, "yy", False, "suffix2_nogeo")

    dedup = {}
    for item in candidates:
        key = (item["prefix"], item["geo_code"], item["date_token"], item["date_kind"], item["uses_geo"])
        if key not in dedup or item["score"] > dedup[key]["score"]:
            dedup[key] = item
    return sorted(dedup.values(), key=lambda item: (-item["score"], item["prefix"], item["date_kind"]))


def choose_best_parse(stem: str) -> dict[str, Any] | None:
    candidates = candidate_parses(stem)
    return candidates[0] if candidates else None


def build_inventory_row(scan_row: dict[str, Any]) -> dict[str, Any] | None:
    full_path = normalize_public_path(str(scan_row.get("full_path") or ""))
    if not full_path or scan_row.get("entry_type") == "dir":
        return None

    pure = PurePosixPath(full_path)
    stem = strip_all_suffixes(pure.name)
    context = infer_system_context(full_path)
    best = choose_best_parse(stem)

    return {
        "path": full_path,
        "directory": str(pure.parent),
        "filename": pure.name,
        "stem": stem,
        "extension_chain": "".join(suffix_chain(full_path)),
        "primary_extension": primary_extension(full_path),
        "format_family": format_family(full_path),
        "asset_kind": classify_asset(full_path),
        "distribution_root": context["distribution_root"],
        "system": context["system"],
        "subsystem": context["subsystem"],
        "context_parts": context["context_parts"],
        "meaningful_context_parts": context["meaningful_context_parts"],
        "path_semantic": infer_path_semantic(full_path),
        "parse": best,
        "parse_candidates": candidate_parses(stem),
    }

def tokenize_text(text: str | None) -> set[str]:
    if not text:
        return set()
    upper = str(text).upper()
    parts = [token for token in TOKEN_SPLIT_RE.split(upper) if token]
    return {token for token in parts if len(token) >= 2}


def infer_path_semantic(path: str) -> str:
    upper_parts = [part.upper() for part in split_public_parts(path)]
    if any(token in {"TESTE", "TESTES"} for token in upper_parts):
        return "[Test]"
    if any(token in {"PRELIM", "PRELIMINAR", "HOMOLOG", "HOMOLOGACAO"} for token in upper_parts):
        return "[Staging]"
    if any(token in {"FINAIS", "FINAL"} for token in upper_parts):
        return "[Primary]"
    return "[Primary]"


def file_quality_key(row: dict[str, Any]) -> tuple:
    ext = row.get("primary_extension")
    semantic = row.get("path_semantic") or "[Primary]"
    return (
        FORMAT_PREFERENCE.get(ext, 99),
        PATH_SEMANTIC_RANK.get(semantic, 99),
        len(str(row.get("path") or "")),
        str(row.get("path") or ""),
    )


def choose_primary_date_kind(rows: list[dict[str, Any]]) -> str | None:
    counts = Counter((row.get("parse") or {}).get("date_kind") for row in rows if (row.get("parse") or {}).get("date_kind"))
    if not counts:
        return None
    return sorted(counts, key=lambda kind: (-counts[kind], DATE_KIND_RANK.get(kind, 99), kind))[0]


def looks_like_implicit_national(rows: list[dict[str, Any]]) -> bool:
    periods = [(row.get("parse") or {}).get("period") for row in rows]
    periods = [period for period in periods if period is not None]
    if len(set(periods)) < 2:
        return False
    if any((row.get("parse") or {}).get("uses_geo") for row in rows):
        return False
    counts = Counter(periods)
    return max(counts.values()) == 1


def common_path_prefix(paths: list[str]) -> str | None:
    if not paths:
        return None
    parts_list = [list(PurePosixPath(path).parts) for path in sorted(set(paths))]
    common = []
    for tokens in zip(*parts_list):
        if len(set(tokens)) == 1:
            common.append(tokens[0])
        else:
            break
    return PurePosixPath(*common).as_posix() if common else None


def build_physical_variants(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_variant = defaultdict(list)
    for row in rows:
        parse = row.get("parse") or {}
        key = (
            parse.get("date_kind"),
            bool(parse.get("uses_geo")),
            row.get("distribution_root"),
            row.get("subsystem"),
            row.get("format_family"),
        )
        by_variant[key].append(row)

    variants = []
    for index, (key, variant_rows) in enumerate(sorted(by_variant.items()), start=1):
        periods = sorted({(row.get("parse") or {}).get("period") for row in variant_rows if (row.get("parse") or {}).get("period") is not None})
        geo_codes = sorted({(row.get("parse") or {}).get("geo_code") for row in variant_rows if (row.get("parse") or {}).get("geo_code")})
        variants.append({
            "variant_id": f"variant_{index}",
            "date_kind": key[0],
            "uses_geo": key[1],
            "distribution_root": key[2],
            "subsystem": key[3],
            "format_family": key[4],
            "file_count": len(variant_rows),
            "period_count": len(periods),
            "time_range_display": (
                f"{period_label(min(periods))} to {period_label(max(periods))}"
                if periods and min(periods) != max(periods)
                else period_label(periods[0]) if periods else None
            ),
            "geo_coverage": geo_codes,
            "source_paths": sorted({row["directory"] for row in variant_rows}),
            "primary_extensions": sorted({row["primary_extension"] for row in variant_rows if row.get("primary_extension")}),
            "sample_files": sorted(row["path"] for row in variant_rows)[:12],
        })
    return variants


def build_state_period_panels(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    selected = {}
    for row in rows:
        parse = row.get("parse") or {}
        period = parse.get("period")
        geo_code = parse.get("geo_code")
        if period is None or geo_code not in UF_CODES:
            continue
        key = (period, geo_code)
        current = selected.get(key)
        if current is None or file_quality_key(row) < file_quality_key(current):
            selected[key] = row

    by_period = defaultdict(list)
    for (_, _), row in selected.items():
        by_period[(row.get("parse") or {}).get("period")].append(row)

    period_panels = []
    for period in sorted(by_period):
        panel_rows = sorted(by_period[period], key=lambda item: ((item.get("parse") or {}).get("geo_code") or "", item["path"]))
        territories = sorted({(item.get("parse") or {}).get("geo_code") for item in panel_rows if (item.get("parse") or {}).get("geo_code")})
        period_panels.append({
            "period": period,
            "period_label": period_label(period),
            "file_count": len(panel_rows),
            "territories": territories,
            "coverage_ratio_ufs": round(len(set(territories) & UF_CODES) / 27, 4),
            "missing_ufs": sorted(UF_CODES - set(territories)),
            "files": [item["path"] for item in panel_rows],
        })
    return period_panels


def build_single_track(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    selected = {}
    for row in rows:
        parse = row.get("parse") or {}
        period = parse.get("period")
        if period is None:
            continue
        current = selected.get(period)
        if current is None or file_quality_key(row) < file_quality_key(current):
            selected[period] = row
    return [selected[period] for period in sorted(selected)]


def summarize_gap_list(gaps: list[dict[str, Any]], *, date_kind: str, period_panel_count: int) -> dict[str, Any]:
    missing_periods = [gap["period"] for gap in gaps if gap.get("kind") == "missing_period" and isinstance(gap.get("period"), int)]
    partial_panels = [gap for gap in gaps if gap.get("kind") == "partial_period_panel"]

    missing_uf_counter = Counter()
    territory_counts = []
    for gap in partial_panels:
        missing_uf_counter.update(gap.get("missing_ufs") or [])
        territory_counts.append(int(gap.get("territory_count") or 0))

    return {
        "gap_count": len(gaps),
        "missing_period_count": len(missing_periods),
        "partial_period_panel_count": len(partial_panels),
        "partial_panel_territory_count_min": min(territory_counts) if territory_counts else None,
        "partial_panel_territory_count_max": max(territory_counts) if territory_counts else None,
        "top_missing_ufs": missing_uf_counter.most_common(12),
        "period_panel_count": period_panel_count,
    }

def coalesce_special_prefixes(data_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_system_kind = defaultdict(set)
    for row in data_rows:
        parse = row.get("parse") or {}
        prefix = parse.get("prefix")
        if prefix:
            by_system_kind[(row.get("system"), parse.get("date_kind"), bool(parse.get("uses_geo")))].add(prefix)

    for row in data_rows:
        parse = row.get("parse") or {}
        prefix = parse.get("prefix")
        if not prefix or parse.get("uses_geo"):
            continue
        family_prefixes = by_system_kind[(row.get("system"), parse.get("date_kind"), True)]
        if len(prefix) >= 5:
            short = prefix[:-2]
            if short in family_prefixes:
                patched = dict(parse)
                patched["prefix_coalesced_from"] = prefix
                patched["prefix"] = short
                row["parse"] = patched
    return data_rows


def infer_partition_type(geo_codes: set[str], *, grammar_uses_geo: bool, rows: list[dict[str, Any]]) -> tuple[str, str]:
    geo_codes = {code for code in geo_codes if code}
    has_br = "BR" in geo_codes
    has_uf = bool(geo_codes & UF_CODES)

    if has_br and has_uf:
        return "Mixed-Partition", "explicit_br_and_ufs"
    if has_uf:
        return "State-Partitioned", "explicit_ufs"
    if has_br:
        return "Nation-Wide", "explicit_br"
    if looks_like_implicit_national(rows):
        return "Nation-Wide", "implicit_single_file_per_period"
    if grammar_uses_geo:
        return "Geo-Coded-Unresolved", "geo_expected_but_unresolved"
    return "Geo-Less", "no_geo_in_filename"


def infer_coverage_status(geo_codes: set[str], partition_type: str) -> str:
    geo_codes = {code for code in geo_codes if code in UF_CODES or code == "BR"}
    if partition_type == "State-Partitioned":
        count = len(geo_codes & UF_CODES)
        if count >= 27:
            return "complete_state_panel"
        if count >= 20:
            return "broad_state_panel"
        if count >= 5:
            return "partial_state_panel"
        return "sparse_state_panel"
    if partition_type == "Nation-Wide":
        return "nationwide"
    if partition_type == "Mixed-Partition":
        return f"mixed_br_plus_{len(geo_codes & UF_CODES)}_ufs"
    if partition_type == "Geo-Less":
        return "geo_less"
    return "unresolved"


def build_doc_row(row: dict[str, Any]) -> dict[str, Any]:
    path = row["path"]
    filename = row["filename"]
    tokens = set(part.lower() for part in split_public_parts(path))
    name_lower = filename.lower()
    keyword_hits = sorted({kw for kw in KEYWORD_DOC_HINTS if kw in name_lower})
    return {
        "path": path,
        "url": f"ftp://ftp.datasus.gov.br{path}",
        "filename": filename,
        "directory": row["directory"],
        "system": row["system"],
        "subsystem": row["subsystem"],
        "distribution_root": row["distribution_root"],
        "format_family": row["format_family"],
        "doc_scope_hint": (
            "system" if any(tok in {"docs", "doc"} for tok in tokens) and row["subsystem"] is None
            else "subsystem" if row["subsystem"]
            else "generic"
        ),
        "keyword_hits": keyword_hits,
    }


def build_logical_families(data_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_key = defaultdict(list)
    for row in data_rows:
        parse = row.get("parse") or {}
        prefix = parse.get("prefix")
        if not prefix or not row.get("system"):
            continue
        by_key[(row["system"], prefix)].append(row)

    families = []
    for (system, prefix), rows in sorted(by_key.items()):
        primary_date_kind = choose_primary_date_kind(rows)
        primary_rows = [row for row in rows if (row.get("parse") or {}).get("date_kind") == primary_date_kind] if primary_date_kind else list(rows)

        geo_codes = {((row.get("parse") or {}).get("geo_code")) for row in primary_rows if (row.get("parse") or {}).get("geo_code")}
        grammar_uses_geo = any(bool((row.get("parse") or {}).get("uses_geo")) for row in primary_rows)
        partition_type, partition_evidence = infer_partition_type(geo_codes, grammar_uses_geo=grammar_uses_geo, rows=primary_rows)
        coverage_status = infer_coverage_status(geo_codes, partition_type)

        periods = sorted({(row.get("parse") or {}).get("period") for row in primary_rows if (row.get("parse") or {}).get("period") is not None})
        period_min = min(periods) if periods else None
        period_max = max(periods) if periods else None

        if partition_type == "State-Partitioned":
            period_panels = build_state_period_panels(primary_rows)
            expected = expected_periods(period_min, period_max, primary_date_kind) if primary_date_kind and period_min is not None and period_max is not None else []
            panel_index = {panel["period"]: panel for panel in period_panels}
            gaps = []
            for period in expected:
                panel = panel_index.get(period)
                if panel is None:
                    gaps.append({"kind": "missing_period", "period": period, "period_label": period_label(period)})
                elif panel["missing_ufs"]:
                    gaps.append({
                        "kind": "partial_period_panel",
                        "period": period,
                        "period_label": panel["period_label"],
                        "territory_count": len(panel["territories"]),
                        "missing_ufs": panel["missing_ufs"],
                    })
            preferred = {
                "mode": "state_panel",
                "coverage_complete": not gaps,
                "selected_period_panels": period_panels,
                "gaps": gaps,
            }
        else:
            selected_track = build_single_track(primary_rows)
            observed_periods = {(row.get("parse") or {}).get("period") for row in selected_track if (row.get("parse") or {}).get("period") is not None}
            expected = expected_periods(period_min, period_max, primary_date_kind) if primary_date_kind and period_min is not None and period_max is not None else []
            gaps = [
                {"kind": "missing_period", "period": period, "period_label": period_label(period)}
                for period in expected if period not in observed_periods
            ]
            preferred = {
                "mode": "national_series" if partition_type == "Nation-Wide" else "geo_less_series",
                "coverage_complete": not gaps if partition_type == "Nation-Wide" else None,
                "selected_files": [row["path"] for row in selected_track],
                "selected_file_count": len(selected_track),
                "gaps": gaps,
            }
            period_panels = []

        subsystem_counts = Counter(row.get("subsystem") for row in rows if row.get("subsystem"))
        subsystem = subsystem_counts.most_common(1)[0][0] if subsystem_counts else None

        families.append({
            "family_id": f"{system}:{prefix}",
            "system": system,
            "distribution_roots": sorted({row.get("distribution_root") for row in rows if row.get("distribution_root")}),
            "subsystem": subsystem,
            "series_prefix": prefix,
            "date_kind": primary_date_kind,
            "available_date_kinds": sorted({(row.get("parse") or {}).get("date_kind") for row in rows if (row.get("parse") or {}).get("date_kind")}),
            "partition_type": partition_type,
            "partition_evidence": partition_evidence,
            "coverage_status": coverage_status,
            "geo_coverage": sorted(code for code in geo_codes if code),
            "file_count": len(rows),
            "time_range_display": (
                f"{period_label(period_min)} to {period_label(period_max)}"
                if period_min is not None and period_max is not None and period_min != period_max
                else period_label(period_min)
            ),
            "source_paths": sorted({row["directory"] for row in rows}),
            "format_families": sorted({row["format_family"] for row in rows}),
            "primary_extensions": sorted({row["primary_extension"] for row in rows if row["primary_extension"]}),
            "member_files": sorted(row["path"] for row in rows),
            "period_panel_count": len(period_panels),
            "period_panels": period_panels,
            "physical_variants": build_physical_variants(rows),
            "preferred_working_series": preferred,
        })
    return families


def associate_docs(families: list[dict[str, Any]], doc_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    for family in families:
        fam_system = str(family.get("system") or "").upper()
        fam_subsystem = str(family.get("subsystem") or "").upper()
        fam_prefix = str(family.get("series_prefix") or "").upper()

        fam_tokens = set()
        fam_tokens |= tokenize_text(fam_subsystem)
        if len(fam_prefix) >= 3:
            fam_tokens.add(fam_prefix)
        for path in family.get("source_paths") or []:
            tail = PurePosixPath(path).name
            fam_tokens |= tokenize_text(tail)

        matches = []
        for doc in doc_rows:
            doc_system = str(doc.get("system") or "").upper()
            if doc_system != fam_system:
                continue

            filename_tokens = tokenize_text(strip_all_suffixes(doc.get("filename") or ""))
            dir_tokens = tokenize_text(doc.get("directory") or "")
            doc_tokens = filename_tokens | dir_tokens

            subsystem_match = bool(fam_subsystem) and fam_subsystem == str(doc.get("subsystem") or "").upper()
            prefix_match = len(fam_prefix) >= 3 and fam_prefix in doc_tokens
            overlap = fam_tokens & doc_tokens
            generic_hint = bool(DOC_GENERIC_HINTS & doc_tokens) or bool(doc.get("keyword_hits"))

            tier = None
            score = 0

            if subsystem_match or prefix_match or len(overlap) >= 2:
                tier = "family_specific"
                score = 60
                score += 20 if subsystem_match else 0
                score += 20 if prefix_match else 0
                score += min(len(overlap) * 6, 18)
            elif generic_hint:
                tier = "system_generic"
                score = 35 + min(len(overlap) * 4, 12)
            else:
                continue

            matches.append({
                "url": doc["url"],
                "path": doc["path"],
                "filename": doc["filename"],
                "score": score,
                "match_tier": tier,
                "doc_scope_hint": doc["doc_scope_hint"],
                "keyword_hits": doc["keyword_hits"],
            })

        family_specific = sorted(
            [item for item in matches if item["match_tier"] == "family_specific"],
            key=lambda item: (-item["score"], item["filename"])
        )[:12]
        system_generic = sorted(
            [item for item in matches if item["match_tier"] == "system_generic"],
            key=lambda item: (-item["score"], item["filename"])
        )[:6]

        family["associated_docs"] = family_specific + system_generic
    return families


def summarize_families(families: list[dict[str, Any]]) -> dict[str, Any]:
    return {
        "family_count": len(families),
        "families_by_system": dict(Counter(f["system"] for f in families).most_common()),
        "families_by_partition_type": dict(Counter(f["partition_type"] for f in families).most_common()),
        "families_by_coverage_status": dict(Counter(f["coverage_status"] for f in families).most_common()),
        "families_with_docs": sum(1 for f in families if f.get("associated_docs")),
        "families_without_docs": sum(1 for f in families if not f.get("associated_docs")),
    }


def build_audit(families: list[dict[str, Any]], doc_rows: list[dict[str, Any]]) -> dict[str, Any]:
    matched_doc_paths = {doc["path"] for family in families for doc in family.get("associated_docs", [])}
    orphan_docs = [doc for doc in doc_rows if doc["path"] not in matched_doc_paths]
    problem_families = []

    for family in families:
        issues = []
        pref = family.get("preferred_working_series") or {}
        gaps = list(pref.get("gaps") or [])
        gap_summary = summarize_gap_list(
            gaps,
            date_kind=str(family.get("date_kind") or "yymm"),
            period_panel_count=int(family.get("period_panel_count") or 0),
        )

        if family["partition_type"] == "State-Partitioned":
            if gap_summary["missing_period_count"] > 0:
                issues.append({
                    "code": "missing_periods",
                    "severity": "error",
                    "summary": f'{gap_summary["missing_period_count"]} missing period(s)',
                    "detail": gap_summary,
                })

            if gap_summary["partial_period_panel_count"] > 0:
                min_count = gap_summary["partial_panel_territory_count_min"] or 0
                partial_count = gap_summary["partial_period_panel_count"]
                if family["coverage_status"] in {"partial_state_panel", "sparse_state_panel"} or min_count <= 10:
                    code = "state_panel_fragmented"
                    severity = "error"
                elif partial_count <= 6 and min_count >= 24:
                    code = "minor_panel_defects"
                    severity = "info"
                else:
                    code = "state_panel_incomplete"
                    severity = "warning"
                issues.append({
                    "code": code,
                    "severity": severity,
                    "summary": f'{partial_count} partial panel period(s)',
                    "detail": gap_summary,
                })

            if family["coverage_status"] != "complete_state_panel":
                issues.append({
                    "code": "incomplete_statewide_coverage",
                    "severity": "warning" if family["coverage_status"] == "broad_state_panel" else "error",
                    "summary": f'coverage_status={family["coverage_status"]}',
                    "detail": {
                        "coverage_status": family["coverage_status"],
                        "geo_coverage": family["geo_coverage"],
                        "missing_ufs_overall": sorted(UF_CODES - set(family["geo_coverage"])),
                        "gap_summary": gap_summary,
                    },
                })

        elif family["partition_type"] == "Nation-Wide":
            if gap_summary["missing_period_count"] > 0:
                issues.append({
                    "code": "national_series_gaps",
                    "severity": "warning" if gap_summary["missing_period_count"] <= 6 else "error",
                    "summary": f'{gap_summary["missing_period_count"]} missing national period(s)',
                    "detail": gap_summary,
                })

        elif family["partition_type"] in {"Geo-Coded-Unresolved", "Geo-Less"}:
            issues.append({
                "code": "partition_unresolved",
                "severity": "warning",
                "summary": f'partition_type={family["partition_type"]}',
                "detail": {
                    "partition_type": family["partition_type"],
                    "partition_evidence": family.get("partition_evidence"),
                },
            })

        docs = list(family.get("associated_docs") or [])
        if not docs:
            issues.append({
                "code": "no_associated_docs",
                "severity": "warning",
                "summary": "no associated docs",
                "detail": {"source_paths": family["source_paths"]},
            })
        elif all(doc.get("match_tier") == "system_generic" for doc in docs[: min(3, len(docs))]):
            issues.append({
                "code": "docs_generic_only",
                "severity": "info",
                "summary": "only generic system-level docs matched",
                "detail": {"associated_docs_top": docs[:8]},
            })

        if issues:
            problem_families.append({
                "family_id": family["family_id"],
                "series_prefix": family["series_prefix"],
                "system": family["system"],
                "subsystem": family.get("subsystem"),
                "partition_type": family["partition_type"],
                "coverage_status": family["coverage_status"],
                "time_range_display": family["time_range_display"],
                "issues": issues,
            })

    return {
        "family_count": len(families),
        "problem_family_count": len(problem_families),
        "problem_families": problem_families,
        "orphan_doc_count": len(orphan_docs),
        "orphan_docs_top_200": orphan_docs[:200],
    }


def family_lookup(families: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    return {family["family_id"]: family for family in families}

In [ ]:
scan_rows = read_jsonl(SCAN_JSONL)
inventory_rows = [row for row in (build_inventory_row(item) for item in scan_rows) if row is not None]

data_rows = [row for row in inventory_rows if row["asset_kind"] == "data" and row.get("parse")]
data_rows = coalesce_special_prefixes(data_rows)

doc_rows = [build_doc_row(row) for row in inventory_rows if row["asset_kind"] == "doc"]

write_jsonl(INVENTORY_JSONL, inventory_rows)
write_json(DOC_ASSETS_JSON, doc_rows)

print("scan rows        :", len(scan_rows))
print("inventory rows   :", len(inventory_rows))
print("data asset rows  :", len(data_rows))
print("doc asset rows   :", len(doc_rows))
print()
print("top systems:", Counter(row["system"] for row in data_rows if row["system"]).most_common(15))
print("top date kinds:", Counter((row.get("parse") or {}).get("date_kind") for row in data_rows).most_common())
print("top geo-cue flags:", Counter(bool((row.get("parse") or {}).get("uses_geo")) for row in data_rows).most_common())

scan rows        : 156833
inventory rows   : 156474
data asset rows  : 153487
doc asset rows   : 85

top systems: [('SIASUS', 56573), ('CNES', 47043), ('SIHSUS', 29231), ('SISCAN', 5846), ('CIHA', 4674), ('SINAN', 2826), ('SINASC', 1723), ('PNI', 1504), ('SIM', 1459), ('SISPRENATAL', 944), ('CIH', 869), ('PCE', 427), ('RESP', 280), ('IBGE', 71), ('painel_oncologia', 14)]
top date kinds: [('yymm', 146044), ('yy', 5813), ('yyyy', 1630)]
top partition cues: [(True, 153043), (False, 444)]


In [8]:
families = build_logical_families(data_rows)
families = associate_docs(families, doc_rows)

family_summary = summarize_families(families)
family_audit = build_audit(families, doc_rows)

write_json(FAMILIES_JSON, families)
write_json(FAMILY_SUMMARY_JSON, family_summary)
write_json(FAMILY_AUDIT_JSON, family_audit)

print(json.dumps(family_summary, ensure_ascii=False, indent=2))
print()
print("problem families:", family_audit["problem_family_count"])
print("orphan docs     :", family_audit["orphan_doc_count"])

{
  "family_count": 126,
  "families_by_system": {
    "SINAN": 58,
    "SIASUS": 15,
    "SISCAN": 12,
    "SIM": 10,
    "CNES": 8,
    "SIHSUS": 8,
    "SINASC": 5,
    "PNI": 2,
    "CIH": 1,
    "CIHA": 1,
    "ESUSNOTIFICA": 1,
    "IBGE": 1,
    "PCE": 1,
    "RESP": 1,
    "SISPRENATAL": 1,
    "painel_oncologia": 1
  },
  "families_by_partition_type": {
    "Nation-Wide": 63,
    "State-Partitioned": 36,
    "Geo-Less": 18,
    "Mixed-Partition": 9
  },
  "families_by_coverage_status": {
    "nationwide": 63,
    "complete_state_panel": 25,
    "geo_less": 18,
    "mixed_br_plus_27_ufs": 8,
    "broad_state_panel": 5,
    "partial_state_panel": 3,
    "sparse_state_panel": 3,
    "mixed_br_plus_19_ufs": 1
  },
  "families_with_docs": 126,
  "families_without_docs": 0
}

problem families: 52
orphan docs     : 1


In [9]:
from pprint import pprint

FAMILY_INDEX = family_lookup(families)

def show_family(family_id: str) -> None:
    family = FAMILY_INDEX[family_id]
    print("=" * 100)
    print(family_id)
    print("=" * 100)
    pprint({
        "system": family["system"],
        "subsystem": family["subsystem"],
        "series_prefix": family["series_prefix"],
        "partition_type": family["partition_type"],
        "coverage_status": family["coverage_status"],
        "date_kind": family["date_kind"],
        "time_range_display": family["time_range_display"],
        "file_count": family["file_count"],
        "geo_coverage": family["geo_coverage"],
        "source_paths": family["source_paths"],
        "format_families": family["format_families"],
        "primary_extensions": family["primary_extensions"],
    })
    print()
    print("preferred_working_series")
    pprint(family["preferred_working_series"])
    print()
    print("associated_docs_top_10")
    pprint(family.get("associated_docs", [])[:10])

# Adjust these family ids as you inspect the notebook outputs.
for family_id in ["SIASUS:AB", "SIASUS:AMP", "SIASUS:AN", "SINASC:DNR", "SIM:DOR", "SIM:DOREXT"]:
    if family_id in FAMILY_INDEX:
        show_family(family_id)

SIASUS:AB
{'coverage_status': 'partial_state_panel',
 'date_kind': 'yymm',
 'file_count': 635,
 'format_families': ['dbase'],
 'geo_coverage': ['AC',
                  'AM',
                  'BA',
                  'CE',
                  'DF',
                  'ES',
                  'MA',
                  'MG',
                  'PA',
                  'PB',
                  'PE',
                  'PR',
                  'RN',
                  'RS',
                  'SC',
                  'SE',
                  'SP',
                  'TO'],
 'partition_type': 'State-Partitioned',
 'primary_extensions': ['.dbc'],
 'series_prefix': 'AB',
 'source_paths': ['/dissemin/publicos/SIASUS/200801_/Dados'],
 'subsystem': 'ABAC2501.dbc',
 'system': 'SIASUS',
 'time_range_display': 'Jan 2008 to Jul 2025'}

preferred_working_series
{'coverage_complete': False,
 'gaps': [{'kind': 'partial_period_panel',
           'missing_ufs': ['AC',
                           'AL',
                    

In [10]:
import json
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any


# --- load from memory if available, otherwise from notebook artifacts ---
def _load_json_if_needed(name: str, fallback_path: Path):
    if name in globals():
        return globals()[name]
    return json.loads(fallback_path.read_text(encoding="utf-8"))


families_ = _load_json_if_needed("families", FAMILIES_JSON)
family_summary_ = _load_json_if_needed("family_summary", FAMILY_SUMMARY_JSON)
family_audit_ = _load_json_if_needed("family_audit", FAMILY_AUDIT_JSON)
doc_rows_ = _load_json_if_needed("doc_rows", DOC_ASSETS_JSON)


# --- compact helpers ---
def _period_int_from_gap(gap: dict[str, Any]) -> int | None:
    value = gap.get("period")
    return int(value) if isinstance(value, int) else None


def _period_label_from_int(value: int | None) -> str | None:
    if value is None:
        return None
    if "period_label" in globals():
        return period_label(value)
    year, month = divmod(value, 100)
    if month == 0:
        return str(year)
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    return f"{month_names[month - 1]} {year}" if 1 <= month <= 12 else f"{year}-{month:02d}"


def _next_period_int(value: int, date_kind: str) -> int:
    if "next_period" in globals():
        return next_period(value, date_kind)
    year, month = divmod(value, 100)
    if date_kind == "yymm":
        month += 1
        if month > 12:
            year += 1
            month = 1
        return year * 100 + month
    return (year + 1) * 100


def _compress_missing_periods(periods: list[int], date_kind: str) -> list[dict[str, Any]]:
    if not periods:
        return []
    periods = sorted(set(periods))
    out = []
    start = periods[0]
    prev = periods[0]
    count = 1
    for value in periods[1:]:
        if value == _next_period_int(prev, date_kind):
            prev = value
            count += 1
            continue
        out.append({
            "start": _period_label_from_int(start),
            "end": _period_label_from_int(prev),
            "count": count,
        })
        start = prev = value
        count = 1
    out.append({
        "start": _period_label_from_int(start),
        "end": _period_label_from_int(prev),
        "count": count,
    })
    return out


def _compact_gap_detail(family: dict[str, Any]) -> dict[str, Any]:
    pref = dict(family.get("preferred_working_series") or {})
    gaps = list(pref.get("gaps") or [])
    if not gaps:
        return {"gap_count": 0}

    date_kind = str(family.get("date_kind") or "yymm")
    missing_periods = [gap["period"] for gap in gaps if gap.get("kind") == "missing_period" and isinstance(gap.get("period"), int)]
    partial_panels = [gap for gap in gaps if gap.get("kind") == "partial_period_panel"]

    missing_uf_counter = Counter()
    territory_counts = []
    partial_examples = []
    for gap in partial_panels:
        missing = list(gap.get("missing_ufs") or [])
        missing_uf_counter.update(missing)
        territory_counts.append(int(gap.get("territory_count") or 0))
        if len(partial_examples) < 12:
            partial_examples.append({
                "period": gap.get("period_label"),
                "territory_count": gap.get("territory_count"),
                "missing_ufs": missing,
            })

    return {
        "gap_count": len(gaps),
        "missing_period_count": len(missing_periods),
        "missing_period_ranges": _compress_missing_periods(missing_periods, date_kind),
        "partial_period_panel_count": len(partial_panels),
        "partial_panel_missing_ufs_top": missing_uf_counter.most_common(12),
        "partial_panel_territory_count_min": min(territory_counts) if territory_counts else None,
        "partial_panel_territory_count_max": max(territory_counts) if territory_counts else None,
        "partial_panel_examples": partial_examples,
    }


def _compress_paths(paths: list[str], limit: int = 8) -> dict[str, Any]:
    if not paths:
        return {"count": 0, "common_prefix": None, "tails": []}
    parts_list = [list(PurePosixPath(path).parts) for path in sorted(set(paths))]
    common = []
    for tokens in zip(*parts_list):
        if len(set(tokens)) == 1:
            common.append(tokens[0])
        else:
            break
    common_prefix = PurePosixPath(*common).as_posix() if common else ""
    tails = []
    for path in sorted(set(paths))[:limit]:
        p = PurePosixPath(path)
        tail_parts = p.parts[len(common):]
        tails.append(PurePosixPath(*tail_parts).as_posix() if tail_parts else ".")
    return {
        "count": len(set(paths)),
        "common_prefix": common_prefix or None,
        "tails": tails,
    }


def _compact_docs(docs: list[dict[str, Any]], limit: int = 8) -> list[dict[str, Any]]:
    out = []
    for doc in sorted(docs, key=lambda item: (-float(item.get("score") or 0), str(item.get("filename") or "")))[:limit]:
        out.append({
            "filename": doc.get("filename"),
            "score": doc.get("score"),
            "doc_scope_hint": doc.get("doc_scope_hint"),
            "keyword_hits": doc.get("keyword_hits"),
        })
    return out


def _family_issue_codes(family_row: dict[str, Any]) -> list[str]:
    return [issue.get("code") for issue in family_row.get("issues") or []]


def _family_compact_row(family: dict[str, Any], audit_row: dict[str, Any] | None) -> dict[str, Any]:
    issues = _family_issue_codes(audit_row or {})
    return {
        "family_id": family.get("family_id"),
        "system": family.get("system"),
        "subsystem": family.get("subsystem"),
        "distribution_roots": family.get("distribution_roots"),
        "series_prefix": family.get("series_prefix"),
        "date_kind": family.get("date_kind"),
        "partition_type": family.get("partition_type"),
        "coverage_status": family.get("coverage_status"),
        "time_range_display": family.get("time_range_display"),
        "file_count": family.get("file_count"),
        "period_panel_count": family.get("period_panel_count"),
        "geo_coverage_count": len(family.get("geo_coverage") or []),
        "geo_coverage_sample": list(family.get("geo_coverage") or [])[:12],
        "format_families": family.get("format_families"),
        "primary_extensions": family.get("primary_extensions"),
        "source_paths": _compress_paths(family.get("source_paths") or []),
        "preferred_mode": (family.get("preferred_working_series") or {}).get("mode"),
        "preferred_coverage_complete": (family.get("preferred_working_series") or {}).get("coverage_complete"),
        "gap_summary": _compact_gap_detail(family),
        "associated_doc_count": len(family.get("associated_docs") or []),
        "associated_docs_top": _compact_docs(family.get("associated_docs") or []),
        "issue_codes": issues,
    }


# --- build lookup structures ---
audit_by_family = {
    row["family_id"]: row
    for row in family_audit_.get("problem_families", [])
}

families_by_system = defaultdict(list)
for family in families_:
    families_by_system[str(family.get("system"))].append(family)

problem_families = []
issue_counter = Counter()
issue_to_families = defaultdict(list)
issue_to_systems = defaultdict(Counter)

for family in families_:
    audit_row = audit_by_family.get(family["family_id"])
    compact = _family_compact_row(family, audit_row)
    if compact["issue_codes"]:
        problem_families.append(compact)
        for code in compact["issue_codes"]:
            issue_counter[code] += 1
            issue_to_families[code].append(compact["family_id"])
            issue_to_systems[code][compact["system"]] += 1

orphan_docs = list(family_audit_.get("orphan_docs_top_200") or [])
orphan_by_system = Counter(str(doc.get("system") or "UNKNOWN") for doc in orphan_docs)
orphan_by_scope = Counter(str(doc.get("doc_scope_hint") or "unknown") for doc in orphan_docs)
orphan_by_subsystem = Counter(str(doc.get("subsystem") or "UNKNOWN") for doc in orphan_docs)

# detect families with incomplete statewide coverage even if not already flagged
incomplete_state_panels = []
for family in families_:
    if family.get("partition_type") != "State-Partitioned":
        continue
    if family.get("coverage_status") == "complete_state_panel":
        continue
    incomplete_state_panels.append({
        "family_id": family.get("family_id"),
        "system": family.get("system"),
        "subsystem": family.get("subsystem"),
        "coverage_status": family.get("coverage_status"),
        "geo_coverage_count": len(family.get("geo_coverage") or []),
        "missing_ufs_overall": sorted(set(UF_CODES) - set(family.get("geo_coverage") or [])),
        "gap_summary": _compact_gap_detail(family),
    })

# compact system summaries
system_summaries = []
for system, rows in sorted(families_by_system.items()):
    system_problem_rows = [row for row in problem_families if row["system"] == system]
    system_summaries.append({
        "system": system,
        "family_count": len(rows),
        "problem_family_count": len(system_problem_rows),
        "partition_types": dict(Counter(row.get("partition_type") for row in rows).most_common()),
        "coverage_statuses": dict(Counter(row.get("coverage_status") for row in rows).most_common()),
        "date_kinds": dict(Counter(row.get("date_kind") for row in rows).most_common()),
        "primary_extensions": dict(Counter(ext for row in rows for ext in (row.get("primary_extensions") or [])).most_common()),
        "problem_families_top": [row["family_id"] for row in sorted(system_problem_rows, key=lambda item: (-len(item["issue_codes"]), item["family_id"]))[:20]],
    })

# final compact payload
compact_payload = {
    "meta": {
        "project_root": str(PROJECT_ROOT),
        "scan_jsonl": str(SCAN_JSONL),
        "family_artifact": str(FAMILIES_JSON),
        "summary_artifact": str(FAMILY_SUMMARY_JSON),
        "audit_artifact": str(FAMILY_AUDIT_JSON),
    },
    "global_summary": {
        "family_count": family_summary_.get("family_count"),
        "families_by_system": family_summary_.get("families_by_system"),
        "families_by_partition_type": family_summary_.get("families_by_partition_type"),
        "families_by_coverage_status": family_summary_.get("families_by_coverage_status"),
        "families_with_docs": family_summary_.get("families_with_docs"),
        "families_without_docs": family_summary_.get("families_without_docs"),
        "problem_family_count": family_audit_.get("problem_family_count"),
        "orphan_doc_count": family_audit_.get("orphan_doc_count"),
        "doc_asset_count": len(doc_rows_),
    },
    "issue_overview": {
        code: {
            "count": issue_counter[code],
            "systems": dict(issue_to_systems[code].most_common()),
            "families": sorted(issue_to_families[code]),
        }
        for code in sorted(issue_counter)
    },
    "systems": system_summaries,
    "incomplete_statewide_coverage": incomplete_state_panels,
    "problem_families": sorted(problem_families, key=lambda row: (-len(row["issue_codes"]), row["system"], row["family_id"])),
    "orphan_docs": {
        "count": len(orphan_docs),
        "by_system": dict(orphan_by_system.most_common()),
        "by_subsystem": dict(orphan_by_subsystem.most_common(30)),
        "by_scope_hint": dict(orphan_by_scope.most_common()),
        "top_examples": [
            {
                "system": doc.get("system"),
                "subsystem": doc.get("subsystem"),
                "filename": doc.get("filename"),
                "doc_scope_hint": doc.get("doc_scope_hint"),
                "keyword_hits": doc.get("keyword_hits"),
                "path": doc.get("path"),
            }
            for doc in orphan_docs[:80]
        ],
    },
}

COMPACT_SUMMARY_JSON = NOTEBOOK_ROOT / "datasus_run_compact_summary.json"
write_json(COMPACT_SUMMARY_JSON, compact_payload)

print("Wrote compact run summary ->", COMPACT_SUMMARY_JSON)
print()
print(json.dumps({
    "family_count": compact_payload["global_summary"]["family_count"],
    "problem_family_count": compact_payload["global_summary"]["problem_family_count"],
    "issue_codes": list(compact_payload["issue_overview"].keys()),
    "orphan_doc_count": compact_payload["global_summary"]["orphan_doc_count"],
}, ensure_ascii=False, indent=2))

Wrote compact run summary -> c:\Users\Galaxy\LEVI\projects\PegaSUS\notebooks\data\catalog\notebook_experiments\datasus_run_compact_summary.json

{
  "family_count": 126,
  "problem_family_count": 52,
  "issue_codes": [
    "coverage_gaps",
    "incomplete_statewide_coverage",
    "partition_unresolved"
  ],
  "orphan_doc_count": 1
}
